In [85]:
import os
from collections import Counter
from pathlib import Path
import pandas as pd
import json
from datetime import datetime
import csv


In [ ]:
# extract unique hash_id from each country folders and save to a csv file
country = "united_kingdom/Enforcements"
base_path = f"../documents/{country}"
output_file = "/..documents/unique_hash_ids.csv"

all_hash_ids = []
for item in os.listdir(base_path):
    full_path = os.path.join(base_path, item)
    if os.path.isdir(full_path):
        all_hash_ids.append(item)

total_count = len(all_hash_ids)
counter = Counter(all_hash_ids)
duplicates = [k for k, v in counter.items() if v > 1]
unique_hash_ids = list(counter.keys())

with open(output_file, "a") as f:
    for hash_id in unique_hash_ids:
        f.write(f"{country},{hash_id}\n")


print(f"Country: {country}")
print(f"Total hash_id folders: {total_count}")
print(f"Duplicate hash_id folders: {len(duplicates)}")
if duplicates:
    print("Duplicate names:", duplicates)
print(f"Unique hash IDs saved to {output_file}")



In [4]:
# check how many duplicate hash_id values in total
csv_path = Path("../documents/unique_hash_ids.csv")
df = pd.read_csv(csv_path, names=["country", "hash_id"])

duplicate_hashes = df[df.duplicated(subset=["hash_id"], keep=False)]

num_duplicates = duplicate_hashes["hash_id"].nunique()
total_duplicates = len(duplicate_hashes)

print(f"Total duplicate hash_id entries (counting all rows): {total_duplicates}")
print(f"Number of unique duplicated hash_ids: {num_duplicates}")

print("\nDuplicated hash_ids by country:")
print(duplicate_hashes.sort_values("hash_id").to_string(index=False))


Total duplicate hash_id entries (counting all rows): 66
Number of unique duplicated hash_ids: 31

Duplicated hash_ids by country:
                       country                          hash_id
           denmark/Decisions_2 02aa4a1136bad68192038af2e56cf4e3
             denmark/Decisions 02aa4a1136bad68192038af2e56cf4e3
           denmark/Decisions_2 06560122abdf4d7af1ba0a148dca643f
             denmark/Decisions 06560122abdf4d7af1ba0a148dca643f
           denmark/Decisions_2 0b4105ce8649097e2bac5c4e6e9e4900
             denmark/Decisions 0b4105ce8649097e2bac5c4e6e9e4900
  germany/rhineland_palatinate 0d6ae8cd013715a25d755f3eeac159e9
                germany/berlin 0d6ae8cd013715a25d755f3eeac159e9
               germany/hamburg 0d6ae8cd013715a25d755f3eeac159e9
           denmark/Decisions_2 0f95feb30cdc686be9cca524572ebfb5
             denmark/Decisions 0f95feb30cdc686be9cca524572ebfb5
           denmark/Decisions_2 11fd2831920f61295e7f969f32a54cd5
             denmark/Decisions 11fd283

### Create a new field in metadata.json
### for all metadata.json have 'decision' field (prior 2023-August data)
###  documentType: GDPR or non-GDPR

In [76]:
country = "united_kingdom/Enforcements"
base_path = f"../documents/{country}"
cutoff_date = datetime.strptime("01/08/2023", "%d/%m/%Y")

all_hash_ids = []
for item in os.listdir(base_path):
    full_path = os.path.join(base_path, item)
    if os.path.isdir(full_path):
        metadata_path = os.path.join(full_path, "metadata.json")
        try:
            with open(metadata_path, "r", encoding="utf-8") as f:
                content = json.load(f)
        except json.JSONDecodeError:
            continue
        except FileNotFoundError:
            continue
        
        if not isinstance(content, list):
            try:
                release_date_str = content.get("releaseDate")
            except AttributeError:
                print(f"Error reading releaseDate in {metadata_path}")
                continue
            if release_date_str:
                try:
                    if len(release_date_str) == 4:
                        release_date =  datetime(int(release_date_str), 1, 1)
                    else:
                        release_date = datetime.strptime(release_date_str, "%d/%m/%Y")
                
                except ValueError:
                    continue
                
                if release_date < cutoff_date:
                    print(f"Processing {metadata_path} with release date {release_date_str}")
                    if "decision" in content:
                        content["documentType"] = "GDPR"
                    else:
                        content["documentType"] = "non-GDPR"

                    with open(metadata_path, "w", encoding="utf-8") as f:
                        json.dump(content, f, ensure_ascii=False, indent=4)

                # if release_date >= cutoff_date and "decision" in content :
                #     print(f"After cutoff date {metadata_path} with release date {release_date_str} has 'decision' field")

        if isinstance(content, list):
            for entry in content:
                release_date_str = entry.get("releaseDate")
                try:
                    if len(release_date_str) == 4:
                        release_date =  datetime(int(release_date_str), 1, 1)
                    else:
                        release_date = datetime.strptime(release_date_str, "%d/%m/%Y")
                
                except ValueError:
                    continue

                if release_date < cutoff_date:
                    #print(f"Processing {metadata_path} with release date {release_date_str}")
                    if "decision" in entry:
                        entry["documentType"] = "GDPR"
                    else:
                        entry["documentType"] = "non-GDPR"

                    with open(metadata_path, "w", encoding="utf-8") as f:
                        json.dump(content, f, ensure_ascii=False, indent=4)

Processing ../documents/united_kingdom/Enforcements/6910c0984bc343de108763a860aa6aab/metadata.json with release date 08/10/2020
Processing ../documents/united_kingdom/Enforcements/4e8862a66ecea562312845f1c1952ccb/metadata.json with release date 07/09/2021
Processing ../documents/united_kingdom/Enforcements/ea0315e93d03dd24bfa396455c33d5e3/metadata.json with release date 28/11/2018
Processing ../documents/united_kingdom/Enforcements/34f4e7996749c3f189e51ac2c741b2c8/metadata.json with release date 03/08/2020
Processing ../documents/united_kingdom/Enforcements/85fc311df96bd755f22d22e6fe672e45/metadata.json with release date 31/03/2022
Processing ../documents/united_kingdom/Enforcements/8f68b28256477bf6dd5cda746a53a821/metadata.json with release date 10/09/2020
Processing ../documents/united_kingdom/Enforcements/c01e3146e630c2cde70918ef57bf3724/metadata.json with release date 21/09/2022
Processing ../documents/united_kingdom/Enforcements/a311cbcad1e1d02ee0cbd91874e1913d/metadata.json with 

### for all metadata.json file who have the 'documentType' field (decision type & prior 2023-8)
### How many are documentType = 'GDPR', how many are documentType = 'non-GDPR'

In [88]:
import os
import json
from datetime import datetime

base_path = "../documents"
cutoff_date = datetime.strptime("01/08/2023", "%d/%m/%Y")
results = []

def parse_date(date_str):
    """Safely parse different date formats (day/month/year, ISO, year-only)."""
    if not date_str:
        return None
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    if date_str.isdigit() and len(date_str) == 4:
        return datetime(int(date_str), 1, 1)
    return None


gdpr_count = 0
nongdpr_count = 0
skipped = 0

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file != "metadata.json":
            continue
        metadata_path = os.path.join(root, file)

        try:
            with open(metadata_path, "r", encoding="utf-8") as f:
                content = json.load(f)
        except (json.JSONDecodeError, FileNotFoundError):
            continue
    
        
        if not isinstance(content, list):

            # Only count if "decision" key exists and documentType field exists
            if "documentType" in content:
                if content["documentType"] == "GDPR":
                    gdpr_count += 1
                elif content["documentType"] == "non-GDPR":
                    nongdpr_count += 1
                else:
                    skipped += 1
                results.append({
                "hash_id": content["md5"],
                "documentType": content["documentType"]
                })
        elif isinstance(content, list):
            for entry in content:
                if "documentType" in content:
                    if content["documentType"] == "GDPR":
                        gdpr_count += 1
                    elif content["documentType"] == "non-GDPR":
                        nongdpr_count += 1
                    else:
                        skipped += 1
                if "documentType" in content:
                    results.append({
                        "hash_id": content[0]["md5"],
                        "documentType": content[0]["documentType"]
                    })


print("=== Summary of DocumentType Counts (prior to 2023-08) ===")
print(f"GDPR:      {gdpr_count}")
print(f"non-GDPR:  {nongdpr_count}")
print(f"Skipped (other values): {skipped}")

output_file = "../documents/document_type_summary.csv"


with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["hash_id", "documentType"])
    writer.writeheader()
    writer.writerows(results)


=== Summary of DocumentType Counts (prior to 2023-08) ===
GDPR:      4022
non-GDPR:  4430
Skipped (other values): 0


In [ ]:
### how many new document later than 2023-08
all_enforcements = '../documents/unique_hash_ids.csv'
prior_annotated = '../documents/document_type_summary.csv'
all_enforcements_df = pd.read_csv(all_enforcements)
prior_annotated_df =  pd.read_csv(prior_annotated)
new_column = pd.merge(all_enforcements_df, prior_annotated_df, on= 'hash_id', how ='left')
new_column = new_column.drop_duplicates()
new_column['documentType'] = new_column['documentType'].fillna('needProcess')
new_column.to_csv('../unique_hash_ids_document_type.csv', index=False)